# Product Co-purchase Network Preparation

This notebook prepares the Online Retail II transaction dataset for relational data visualisation.

The final goal is to create two Gephi-ready files:

- `nodes.csv`: a list of products
- `edges.csv`: a list of co-purchase relationships between products

These files will later be imported into Gephi to visualise the product co-purchase network.

## 1. Project Setup and Graph Logic

### 1.1 Project Objective

The objective of this project is to transform an e-commerce transaction dataset into a relational network.

In this network:

- products are represented as nodes;
- co-purchase relationships are represented as edges;
- edge weight shows how frequently two products were purchased together.

This allows the visualisation to reveal product hubs, product clusters, bridge products, and irregular product relationships.

### 1.2 Graph Notation

The product co-purchase network is defined as:

**G = (V, E, w)**

where:

- **V** = the set of product nodes;
- **E** = the set of undirected product-product edges;
- **w(u, v)** = the edge weight between product *u* and product *v*.

For this project:

- each product is represented as one node;
- an edge exists when two products appear together in the same invoice basket;
- the edge weight represents the number of invoices in which the two products were purchased together.

Because co-purchase relationships have no direction, the graph is treated as undirected. This means that if product *u* is purchased with product *v*, then product *v* is also purchased with product *u*.

For the Gephi visualisations, filtered networks are created by applying edge-weight thresholds:

**G₃₀ = (V₃₀, E₃₀, w)**

**G₁₀₀ = (V₁₀₀, E₁₀₀, w)**

where:

- **E₃₀** includes product pairs with edge weight ≥ 30;
- **E₁₀₀** includes product pairs with edge weight ≥ 100;
- **V₃₀** includes products that appear in **E₃₀**;
- **V₁₀₀** includes products that appear in **E₁₀₀**.

Therefore, the threshold-30 and threshold-100 networks have different node and edge sets.

## 2. Data Loading and Inspection

### 2.1 Import Library

Only `pandas` is required at this stage because the first task is to load, inspect, and clean the transaction dataset.

In [1]:
import pandas as pd

### 2.2 Load Dataset

The dataset is loaded from the `data/raw` folder. The file contains transaction-level records from an online retail business.

In [2]:
df = pd.read_csv(
    "../data/raw/online_retail_II.csv",
    encoding="ISO-8859-1",
    dtype={"Invoice": str, "StockCode": str},
    low_memory=False
)

The dataset is loaded from the `data/raw` folder. `Invoice` and `StockCode` are read as text because they are identifiers rather than numerical values. This prevents mixed-type issues and keeps the transaction and product IDs consistent during cleaning and network construction.

### 2.3 Inspect Dataset Size

The shape of the dataset shows the number of rows and columns.

Each row is expected to represent one product line inside an invoice, not one complete customer order.

In [3]:
df.shape

(1067371, 8)

The dataset contains 1,067,371 rows and 8 columns. Each row represents one product line within an invoice, not one complete customer order. This is important because a single invoice can contain multiple products. Therefore, the dataset must later be grouped by Invoice before product co-purchase relationships can be created.

In the context of this project, the transaction table is not yet a graph. It first needs to be transformed into a relational structure where products become nodes and products purchased together in the same invoice become edges.

### 2.4 Inspect Columns and Sample Records

The next step is to check the column names and preview the first few rows. This helps identify which columns will be used for graph construction.

In [4]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [5]:
df.columns

Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'Customer ID', 'Country'],
      dtype='object')

The key columns for this project are:

- `Invoice`: used to group products into the same transaction basket.
- `StockCode`: used as the product ID.
- `Description`: used as the product label.
- `Quantity`: used to calculate product volume.
- `Price`: used to calculate product revenue.
- `InvoiceDate`: used as time context.
- `Customer ID`: used as customer context.
- `Country`: used as market context.

For the network construction, the most important fields are `Invoice`, `StockCode`, and `Description`.

The sample records confirm that the dataset is stored at transaction-line level. This means each row represents one product inside an invoice. For this project, `Invoice`, `StockCode`, and `Description` are the core fields because they allow the raw transaction table to be transformed into a product relationship network.

### 2.5 Check Missing Values

Missing values are checked before cleaning because product IDs, product descriptions, and invoice numbers are required to construct the graph.

In [6]:
df.isna().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

The missing value check shows that Description has 4,382 missing values, while Customer ID has 243,007 missing values.

Rows with missing Description should be removed because product descriptions are needed to interpret product nodes in the network visualisation. However, missing Customer ID values are not removed at this stage because this project does not construct a customer-level network. The network is based on product co-purchase relationships within invoices, so Invoice, StockCode, and Description are the key required fields.

This avoids unnecessarily removing a large number of valid product transaction records only because customer information is missing.

## 3. Data Cleaning

### 3.1 Apply Cleaning Rules

The dataset is cleaned before constructing the network.

The cleaning rules are:

1. Remove cancelled invoices.
2. Keep only rows with positive quantity.
3. Keep only rows with positive price.
4. Remove rows without invoice, product code, or product description.

These rules help ensure that the network is based on valid purchase transactions.

In [7]:
clean_df = df.copy()

# Remove cancelled invoices
clean_df = clean_df[~clean_df["Invoice"].astype(str).str.startswith("C")]

# Keep valid quantity and price
clean_df = clean_df[clean_df["Quantity"] > 0]
clean_df = clean_df[clean_df["Price"] > 0]

# Remove rows with missing key product information
clean_df = clean_df.dropna(subset=["Invoice", "StockCode", "Description"])

clean_df.shape

(1041670, 8)

The cleaning process keeps only valid purchase records. Cancelled invoices are removed because they do not represent completed purchases. Rows with non-positive quantity or price are removed because they may represent returns, corrections, or invalid transactions. Rows with missing invoice, product code, or product description are also removed because they cannot reliably contribute to product nodes or co-purchase edges.

Customer ID is not used as a required cleaning field because the purpose of this project is to construct a product-product network rather than a customer-product or customer-customer network.

### 3.2 Compare Original and Cleaned Dataset

This step checks how many rows were removed during data cleaning.

In [8]:
print("Original rows:", len(df))
print("Cleaned rows:", len(clean_df))
print("Rows removed:", len(df) - len(clean_df))

Original rows: 1067371
Cleaned rows: 1041670
Rows removed: 25701


After cleaning, the dataset was reduced from 1,067,371 rows to 1,041,670 rows. A total of 25,701 rows were removed because they represented cancelled transactions, invalid quantities, invalid prices, or missing key product information.

The cleaned dataset remains large and suitable for network construction. More importantly, it is now more reliable for relational data modelling because the remaining rows represent valid product purchase records that can be grouped into invoice baskets.

### 3.3 Validate Cleaning Results

After cleaning, I validate whether cancelled invoices, invalid quantities, invalid prices, and missing key product fields still remain in the dataset.

In [9]:
print("Cancelled invoices remaining:", clean_df["Invoice"].astype(str).str.startswith("C").sum())
print("Rows with Quantity <= 0:", (clean_df["Quantity"] <= 0).sum())
print("Rows with Price <= 0:", (clean_df["Price"] <= 0).sum())

print("\nMissing values in key fields:")
print(clean_df[["Invoice", "StockCode", "Description"]].isna().sum())

Cancelled invoices remaining: 0
Rows with Quantity <= 0: 0
Rows with Price <= 0: 0

Missing values in key fields:
Invoice        0
StockCode      0
Description    0
dtype: int64


The validation confirms that cancelled invoices, non-positive quantities, non-positive prices, and missing key product fields have been removed. This means the cleaned dataset is ready for the next stage: grouping products by invoice to create transaction baskets.

This validation step is important because any invalid product rows left in the dataset could create misleading nodes or edges in the final co-purchase network.

### 3.4 Save Cleaned Dataset

The cleaned dataset is saved to the `data/processed` folder. This file will be used in the next stage to create invoice baskets, product nodes, and product co-purchase edges.

In [10]:
clean_df.to_csv("../data/processed/online_retail_cleaned.csv", index=False)

In [11]:
clean_df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


## 4. Basket Construction

The next step is to group products by invoice.

In the raw transaction table, each row represents one product line. However, co-purchase relationships can only be identified after products are grouped into transaction baskets.

A basket represents one invoice and contains the unique products purchased in that invoice.

### 4.1 Load Cleaned Dataset

The cleaned dataset is loaded from the `data/processed` folder. This makes the basket construction step reproducible even if the notebook is restarted.

In [12]:
clean_df = pd.read_csv(
    "../data/processed/online_retail_cleaned.csv",
    dtype={"Invoice": str, "StockCode": str},
    low_memory=False
)

clean_df.shape

(1041670, 8)

The cleaned dataset is loaded successfully. `Invoice` and `StockCode` are read as text because they are identifiers rather than numerical values. This keeps transaction IDs and product IDs consistent for basket construction and avoids mixed-type warnings when reading the CSV file.

### 4.2 Group Products by Invoice

Products are grouped by `Invoice` to create transaction baskets. Each basket contains the unique products purchased in the same invoice.

Duplicate product codes within the same invoice are removed because the network only needs to know whether two products appeared together, not how many times the same product appeared in one invoice.

In [13]:
basket_df = clean_df[["Invoice", "StockCode"]].drop_duplicates()

basket_df = basket_df.groupby("Invoice")["StockCode"].apply(list).reset_index()

basket_df = basket_df.rename(columns={"StockCode": "ProductList"})

basket_df["ProductCount"] = basket_df["ProductList"].apply(len)

basket_df.head()

,Invoice,ProductList,ProductCount
0,489434,"[85048, 79323P, 79323W, 22041, 21232, 22064, 2...",8
1,489435,"[22350, 22349, 22195, 22353]",4
2,489436,"[48173C, 21755, 21754, 84879, 22119, 22142, 22...",19
3,489437,"[22143, 22145, 22130, 21364, 21360, 21351, 213...",23
4,489438,"[21329, 21252, 21100, 21033, 20711, 21410, 214...",17


The basket table now contains one row per invoice. `ProductList` shows the unique products in each invoice, while `ProductCount` shows how many unique products are included in that basket.

This table is the bridge between the cleaned transaction data and the final product co-purchase network.

### 4.3 Filter Valid Baskets

Only baskets with at least two unique products can create co-purchase relationships. Single-product invoices are removed because they cannot produce product-product edges.

Very large baskets are also removed because they may represent wholesale or unusual bulk orders. These baskets can create too many product pairs and distort the network structure.

In [14]:
valid_basket_df = basket_df[basket_df["ProductCount"] >= 2]
valid_basket_df = valid_basket_df[valid_basket_df["ProductCount"] <= 50]

valid_basket_df.shape

(32406, 3)

The upper basket-size filter was applied as a visualisation-oriented filtering decision. Very large baskets can generate a disproportionate number of product pairs because a basket with n products creates n(n-1)/2 edges. Therefore, baskets with more than 50 unique products were excluded to reduce distortion from unusually large orders. This improves readability but also means the final network should be interpreted as a filtered exploratory network rather than a complete representation of all transactions.

In [15]:
print("Total invoices:", len(basket_df))
print("Valid baskets:", len(valid_basket_df))
print("Removed baskets:", len(basket_df) - len(valid_basket_df))

Total invoices: 40077
Valid baskets: 32406
Removed baskets: 7671


The basket filtering step reduced the dataset from 40,077 invoices to 32,406 valid baskets. A total of 7,671 baskets were removed because they either contained only one unique product or contained more than 50 unique products.

This filtering is necessary because single-product baskets cannot create co-purchase edges, while extremely large baskets may create too many product pairs and distort the structure of the final network.

### 4.4 Save Basket Data

The valid basket table is saved to the `data/processed` folder. This file will be used in the next stage to generate product-product pairs and calculate edge weights.

In [16]:
basket_export = valid_basket_df.copy()

basket_export["ProductList"] = basket_export["ProductList"].apply(lambda x: "|".join(x))

basket_export.to_csv("../data/processed/invoice_baskets.csv", index=False)

basket_export.head()

,Invoice,ProductList,ProductCount
0,489434,85048|79323P|79323W|22041|21232|22064|21871|21523,8
1,489435,22350|22349|22195|22353,4
2,489436,48173C|21755|21754|84879|22119|22142|22296|222...,19
3,489437,22143|22145|22130|21364|21360|21351|21352|3540...,23
4,489438,21329|21252|21100|21033|20711|21410|21411|8403...,17


The basket file has been saved successfully. Each row represents one invoice basket, and the products inside each basket are separated by the `|` symbol.

This format keeps the basket data readable while making it easy to reuse in the next stage of the notebook.

## 5. Edge Construction

This section creates product co-purchase edges from the valid invoice baskets.

An edge is created when two products appear in the same invoice basket. If the same product pair appears together in multiple invoices, the edge weight increases.

The final output of this section is an edge list that can be imported into Gephi.

### 5.1 Load Basket Data

The valid basket file is loaded from the `data/processed` folder. Each row represents one invoice basket, and the products inside each basket are separated by the `|` symbol.

In [17]:
basket_df = pd.read_csv(
    "../data/processed/invoice_baskets.csv",
    dtype={"Invoice": str}
)

basket_df.head()

,Invoice,ProductList,ProductCount
0,489434,85048|79323P|79323W|22041|21232|22064|21871|21523,8
1,489435,22350|22349|22195|22353,4
2,489436,48173C|21755|21754|84879|22119|22142|22296|222...,19
3,489437,22143|22145|22130|21364|21360|21351|21352|3540...,23
4,489438,21329|21252|21100|21033|20711|21410|21411|8403...,17


The basket file is loaded successfully. At this stage, each invoice has a `ProductList` column containing the products purchased together in that invoice.

### 5.2 Convert Product List Back to List Format

The saved basket file stores products as text separated by the `|` symbol. Before creating product pairs, the product list needs to be converted back into a Python list.

In [18]:
basket_df["ProductList"] = basket_df["ProductList"].apply(lambda x: x.split("|"))

basket_df.head()

,Invoice,ProductList,ProductCount
0,489434,"[85048, 79323P, 79323W, 22041, 21232, 22064, 2...",8
1,489435,"[22350, 22349, 22195, 22353]",4
2,489436,"[48173C, 21755, 21754, 84879, 22119, 22142, 22...",19
3,489437,"[22143, 22145, 22130, 21364, 21360, 21351, 213...",23
4,489438,"[21329, 21252, 21100, 21033, 20711, 21410, 214...",17


The `ProductList` column has been converted back into list format. This allows Python to create all possible product pairs within each invoice basket.

### 5.3 Generate Product Pair Counts

For each invoice basket, all possible pairs of products are generated.

For example, if one basket contains products A, B, and C, then the product pairs are:

- A-B
- A-C
- B-C

Each repeated pair across invoices increases the edge weight by 1.

In [19]:
from itertools import combinations

edge_counts = {}

for product_list in basket_df["ProductList"]:
    product_pairs = combinations(product_list, 2)
    
    for product_a, product_b in product_pairs:
        pair = tuple(sorted([product_a, product_b]))
        
        if pair in edge_counts:
            edge_counts[pair] = edge_counts[pair] + 1
        else:
            edge_counts[pair] = 1

len(edge_counts)

2076826

The output shows that 2,076,826 unique product pairs were found across all valid invoice baskets. Each pair represents one potential edge in the product co-purchase network.

This confirms that the full network is very large. Therefore, edge filtering will be necessary before visualisation to reduce weak relationships and avoid visual clutter in Gephi.

### 5.4 Create Edge Table

The product pair counts are converted into an edge table for Gephi.

In this edge table:

- `Source` is the first product in the pair.
- `Target` is the second product in the pair.
- `Type` is set as `Undirected` because co-purchase relationships have no direction.
- `Weight` is the number of invoices where the two products were purchased together.

In [20]:
edges_df = pd.DataFrame(
    [(source, target, weight) for (source, target), weight in edge_counts.items()],
    columns=["Source", "Target", "Weight"]
)

edges_df["Type"] = "Undirected"

edges_df = edges_df[["Source", "Target", "Type", "Weight"]]

edges_df.head()

,Source,Target,Type,Weight
0,79323P,85048,Undirected,4
1,79323W,85048,Undirected,5
2,22041,85048,Undirected,3
3,21232,85048,Undirected,13
4,22064,85048,Undirected,1


In [21]:
edges_df.sort_values("Weight", ascending=False).head(10)

,Source,Target,Type,Weight
325307,22386,85099B,Undirected,914
1210,21733,85123A,Undirected,805
20748,21931,85099B,Undirected,761
21997,82482,82494L,Undirected,759
16399,85099B,85099F,Undirected,725
8406,20725,20727,Undirected,716
288774,20725,22384,Undirected,714
288090,20725,22383,Undirected,693
325302,22411,85099B,Undirected,688
8379,85099B,85099C,Undirected,670


The edge table now contains product-product co-purchase relationships in a Gephi-compatible format. The highest-weight edges represent product pairs that appeared together in the largest number of invoices.

At this stage, the products are still represented by `StockCode`. Product names will be added later through the node table.

### 5.5 Filter Edges for Gephi

The full edge table contains many weak co-purchase relationships. To make the network readable in Gephi, weak edges are filtered out.

A threshold of 30 is used to create an overview network. This keeps product pairs that appeared together in at least 30 invoices. The overview network is still large, but it is more readable than the full edge table.

In [22]:
edge_threshold = 30

edges_gephi = edges_df[edges_df["Weight"] >= edge_threshold].copy()

edges_gephi.shape

(20065, 4)

In [23]:
edges_gephi.sort_values("Weight", ascending=False).head(10)

,Source,Target,Type,Weight
325307,22386,85099B,Undirected,914
1210,21733,85123A,Undirected,805
20748,21931,85099B,Undirected,761
21997,82482,82494L,Undirected,759
16399,85099B,85099F,Undirected,725
8406,20725,20727,Undirected,716
288774,20725,22384,Undirected,714
288090,20725,22383,Undirected,693
325302,22411,85099B,Undirected,688
8379,85099B,85099C,Undirected,670


The threshold-30 edge table contains 20,065 product-pair relationships. This is much smaller than the full edge table and is more suitable for creating an overview network in Gephi.

However, 20,065 edges may still create visual clutter, so a stricter threshold is also created for focused visual analysis.

### 5.6 Create Strong Edge File for Focused Visualisation

A second, stricter edge file is created using a threshold of 100. This file keeps only very strong co-purchase relationships and will be useful for creating a cleaner network visualisation.

In [24]:
strong_edge_threshold = 100

edges_gephi_strong = edges_df[edges_df["Weight"] >= strong_edge_threshold].copy()

edges_gephi_strong.shape

(1948, 4)

The threshold-100 edge table contains 1,948 product-pair relationships. This is much smaller than the threshold-30 file and will be easier to visualise clearly.

This stricter file is useful for focused analysis of the strongest co-purchase relationships, such as product hubs, strong product clusters, and potential bundle opportunities.

### 5.7 Save Edge Files

The edge files are saved to the `data/processed` folder. These files will later be imported into Gephi together with the node files.

In [25]:
edges_df.to_csv("../data/processed/all_product_edges.csv", index=False)

edges_gephi.to_csv("../data/processed/edges_gephi_threshold30.csv", index=False)

edges_gephi_strong.to_csv("../data/processed/edges_gephi_threshold100.csv", index=False)

In [26]:
print("All edges:", len(edges_df))
print("Threshold 30 edges:", len(edges_gephi))
print("Threshold 100 edges:", len(edges_gephi_strong))

All edges: 2076826
Threshold 30 edges: 20065
Threshold 100 edges: 1948


The edge files have been saved successfully:

- `all_product_edges.csv` keeps all 2,076,826 product-pair relationships.
- `edges_gephi_threshold30.csv` keeps 20,065 stronger relationships for an overview network.
- `edges_gephi_threshold100.csv` keeps 1,948 very strong relationships for focused visualisation.

The filtered files are more suitable for visual analysis because they reduce weak and noisy relationships before importing the data into Gephi.

## 6. Node Construction

This section creates product nodes for the Gephi network.

In the edge file, each relationship uses product codes in the `Source` and `Target` columns. Gephi also needs a node file that defines what each product code represents.

In this project:

- each node represents one product;
- `Id` is the product code;
- `Label` is the product description;
- product-level metrics such as quantity, revenue, degree, and weighted degree are added to support visual analysis.

### 6.1 Load Cleaned Data and Edge Files

The cleaned transaction data is loaded again to create product labels and product-level metrics.

The filtered edge files are also loaded because the node files must match the products that appear in the selected edge files.

In [27]:
clean_df = pd.read_csv(
    "../data/processed/online_retail_cleaned.csv",
    dtype={"Invoice": str, "StockCode": str},
    low_memory=False
)

edges_gephi = pd.read_csv(
    "../data/processed/edges_gephi_threshold30.csv",
    dtype={"Source": str, "Target": str}
)

edges_gephi_strong = pd.read_csv(
    "../data/processed/edges_gephi_threshold100.csv",
    dtype={"Source": str, "Target": str}
)

print("Cleaned data:", clean_df.shape)
print("Threshold 30 edges:", edges_gephi.shape)
print("Threshold 100 edges:", edges_gephi_strong.shape)

Cleaned data: (1041670, 8)
Threshold 30 edges: (20065, 4)
Threshold 100 edges: (1948, 4)


The cleaned transaction data and the two filtered edge files are loaded successfully. The threshold-30 edge file contains 20,065 relationships, while the threshold-100 edge file contains 1,948 relationships.

The node files will be created separately for the threshold-30 network and the threshold-100 network so that each node file matches its corresponding edge file.

### 6.2 Create Product Information Table

The first node-related table stores product information.

`StockCode` is used as the node ID, while `Description` is used as the node label. Duplicate product rows are removed because each product should appear only once in the node table.

In [28]:
product_info = clean_df[["StockCode", "Description"]].drop_duplicates(subset=["StockCode"])

product_info = product_info.rename(columns={
    "StockCode": "Id",
    "Description": "Label"
})

product_info.head()

,Id,Label
0,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS
1,79323P,PINK CHERRY LIGHTS
2,79323W,WHITE CHERRY LIGHTS
3,22041,"RECORD FRAME 7"" SINGLE SIZE"
4,21232,STRAWBERRY CERAMIC TRINKET BOX


The product information table contains one row per product. `Id` will be used by Gephi to match nodes with edges, while `Label` provides a readable product name for the visualisation.

### 6.3 Create Product Sales Metrics

Product-level sales metrics are calculated from the cleaned transaction data.

These metrics are not required for Gephi to draw the graph, but they are useful for interpretation. For example, they help compare whether highly connected products also have high quantity sold or high revenue.

In [29]:
clean_df["Revenue"] = clean_df["Quantity"] * clean_df["Price"]

product_sales = clean_df.groupby("StockCode").agg({
    "Quantity": "sum",
    "Revenue": "sum",
    "Invoice": "nunique"
}).reset_index()

product_sales = product_sales.rename(columns={
    "StockCode": "Id",
    "Quantity": "TotalQuantity",
    "Revenue": "TotalRevenue",
    "Invoice": "InvoiceCount"
})

product_sales.head()

,Id,TotalQuantity,TotalRevenue,InvoiceCount
0,10002,8836,7097.90,362
1,10002R,4,20.57,3
2,10080,315,129.29,27
3,10109,4,1.68,1
4,10120,680,146.10,73


The product sales table shows product-level quantity, revenue, and invoice count. These fields provide additional context for interpreting product nodes in the final network.

### 6.4 Combine Product Information and Sales Metrics

The product label table and product sales table are combined into one product summary table. This table will later be joined with network metrics such as degree and weighted degree.

In [30]:
product_summary = pd.merge(
    product_info,
    product_sales,
    on="Id",
    how="left"
)

product_summary.head()

,Id,Label,TotalQuantity,TotalRevenue,InvoiceCount
0,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,2755,21826.90,517
1,79323P,PINK CHERRY LIGHTS,2281,13787.93,265
2,79323W,WHITE CHERRY LIGHTS,3004,18048.72,349
3,22041,"RECORD FRAME 7"" SINGLE SIZE",7861,19371.46,555
4,21232,STRAWBERRY CERAMIC TRINKET BOX,38981,50760.48,2435


The product summary table now contains both product labels and product-level sales metrics. This table will be used as the base for creating Gephi node files.

### 6.5 Create Node Metrics for Threshold-30 Network

Node metrics are calculated from the threshold-30 edge file.

For each product:

- `Degree` counts how many other products it is connected to.
- `WeightedDegree` sums the weights of all its co-purchase edges.

A product with high degree is connected to many other products. A product with high weighted degree has strong repeated co-purchase relationships.

In [31]:
source_nodes = edges_gephi[["Source", "Weight"]].rename(columns={"Source": "Id"})
target_nodes = edges_gephi[["Target", "Weight"]].rename(columns={"Target": "Id"})

node_connections = pd.concat([source_nodes, target_nodes])

node_metrics = node_connections.groupby("Id").agg({
    "Weight": ["count", "sum"]
}).reset_index()

node_metrics.columns = ["Id", "Degree", "WeightedDegree"]

node_metrics.head()

,Id,Degree,WeightedDegree
0,10002,3,100
1,15034,1,37
2,15036,21,798
3,15044A,4,232
4,15044B,4,216


The node metrics table calculates how connected each product is within the threshold-30 network. These metrics can later be used in Gephi to size nodes or identify product hubs.

### 6.6 Create Node File for Threshold-30 Network

The threshold-30 node metrics are joined with the product summary table. This creates the final node file for the overview network.

In [32]:
nodes_gephi = pd.merge(
    node_metrics,
    product_summary,
    on="Id",
    how="left"
)

nodes_gephi = nodes_gephi[[
    "Id",
    "Label",
    "TotalQuantity",
    "TotalRevenue",
    "InvoiceCount",
    "Degree",
    "WeightedDegree"
]]

nodes_gephi = nodes_gephi.sort_values("WeightedDegree", ascending=False)

nodes_gephi.head(10)

,Id,Label,TotalQuantity,TotalRevenue,InvoiceCount,Degree,WeightedDegree
1621,85123A,WHITE HANGING HEART T-LIGHT HOLDER,96147,263109.67,5365,730,57967
1616,85099B,JUMBO BAG RED WHITE SPOTTY,98349,183454.83,3989,468,40733
733,22423,REGENCY CAKESTAND 3 TIER,27577,344563.25,3918,560,37131
55,20725,LUNCH BAG RED SPOTTY,40942,72292.85,3056,394,31924
163,21212,PACK OF 72 RETRO SPOT CAKE CASES,96560,52997.20,3118,381,26966
1557,84879,ASSORTED COLOUR BIRD ORNAMENT,81809,132187.92,2807,416,25549
710,22383,LUNCHBAG SUKI DESIGN,25461,43932.56,2398,265,21184
713,22386,JUMBO BAG PINK WITH WHITE SPOTS,39840,77111.91,2245,255,21158
437,21931,JUMBO STORAGE BAG SUKI,29503,62285.28,2329,252,20346
57,20727,LUNCH BAG BLACK SKULL.,27126,46422.49,2351,265,20239


In [33]:
print("Threshold 30 nodes:", len(nodes_gephi))
print("Missing labels:", nodes_gephi["Label"].isna().sum())

Threshold 30 nodes: 1653
Missing labels: 0


The threshold-30 node file contains 1,653 products that appear in the threshold-30 edge file. The products are sorted by `WeightedDegree`, so the first rows represent products with the strongest overall co-purchase connections.

The missing label count is 0, which means every product node has a readable product description. This node file is ready to be imported into Gephi with `edges_gephi_threshold30.csv`.

### 6.7 Create Node File for Threshold-100 Network

The same node construction process is repeated for the stricter threshold-100 network. This creates a smaller node file for focused visualisation.

In [34]:
source_nodes_strong = edges_gephi_strong[["Source", "Weight"]].rename(columns={"Source": "Id"})
target_nodes_strong = edges_gephi_strong[["Target", "Weight"]].rename(columns={"Target": "Id"})

node_connections_strong = pd.concat([source_nodes_strong, target_nodes_strong])

node_metrics_strong = node_connections_strong.groupby("Id").agg({
    "Weight": ["count", "sum"]
}).reset_index()

node_metrics_strong.columns = ["Id", "Degree", "WeightedDegree"]

nodes_gephi_strong = pd.merge(
    node_metrics_strong,
    product_summary,
    on="Id",
    how="left"
)

nodes_gephi_strong = nodes_gephi_strong[[
    "Id",
    "Label",
    "TotalQuantity",
    "TotalRevenue",
    "InvoiceCount",
    "Degree",
    "WeightedDegree"
]]

nodes_gephi_strong = nodes_gephi_strong.sort_values("WeightedDegree", ascending=False)

nodes_gephi_strong.head(10)

,Id,Label,TotalQuantity,TotalRevenue,InvoiceCount,Degree,WeightedDegree
575,85123A,WHITE HANGING HEART T-LIGHT HOLDER,96147,263109.67,5365,155,27763
572,85099B,JUMBO BAG RED WHITE SPOTTY,98349,183454.83,3989,89,20749
18,20725,LUNCH BAG RED SPOTTY,40942,72292.85,3056,77,16229
274,22423,REGENCY CAKESTAND 3 TIER,27577,344563.25,3918,77,11879
267,22383,LUNCHBAG SUKI DESIGN,25461,43932.56,2398,51,10923
270,22386,JUMBO BAG PINK WITH WHITE SPOTS,39840,77111.91,2245,43,10634
63,21212,PACK OF 72 RETRO SPOT CAKE CASES,96560,52997.20,3118,56,10195
154,21931,JUMBO STORAGE BAG SUKI,29503,62285.28,2329,42,10111
268,22384,LUNCHBAG PINK RETROSPOT,21037,34506.66,2106,45,9516
20,20727,LUNCH BAG BLACK SKULL.,27126,46422.49,2351,46,9482


In [35]:
print("Threshold 100 nodes:", len(nodes_gephi_strong))
print("Missing labels:", nodes_gephi_strong["Label"].isna().sum())

Threshold 100 nodes: 581
Missing labels: 0


The threshold-100 node file contains 581 products that appear in the strongest co-purchase relationships. The missing label count is 0, so all product nodes have readable labels.

This smaller node file will be used with `edges_gephi_threshold100.csv` to create a cleaner and more focused network in Gephi.

### 6.8 Save Node Files

The node files are saved to the `data/processed` folder. These files will be imported into Gephi together with their matching edge files.

In [36]:
nodes_gephi.to_csv("../data/processed/nodes_gephi_threshold30.csv", index=False)

nodes_gephi_strong.to_csv("../data/processed/nodes_gephi_threshold100.csv", index=False)

In [37]:
print("Saved node files:")
print("- nodes_gephi_threshold30.csv")
print("- nodes_gephi_threshold100.csv")

print("\nNode counts:")
print("Threshold 30 nodes:", len(nodes_gephi))
print("Threshold 100 nodes:", len(nodes_gephi_strong))

Saved node files:
- nodes_gephi_threshold30.csv
- nodes_gephi_threshold100.csv

Node counts:
Threshold 30 nodes: 1653
Threshold 100 nodes: 581


The node files have been saved successfully:

- `nodes_gephi_threshold30.csv` contains 1,653 product nodes for the overview network.
- `nodes_gephi_threshold100.csv` contains 581 product nodes for the focused network.

Together with the matching edge files, the project now has complete Gephi-ready data:

- `nodes_gephi_threshold30.csv` + `edges_gephi_threshold30.csv`
- `nodes_gephi_threshold100.csv` + `edges_gephi_threshold100.csv`

### 6.9 Validate Node and Edge Matching

This step checks whether all products appearing in the edge files are included in the matching node files.

In [38]:
edge_nodes_30 = set(edges_gephi["Source"]) | set(edges_gephi["Target"])
node_ids_30 = set(nodes_gephi["Id"])

edge_nodes_100 = set(edges_gephi_strong["Source"]) | set(edges_gephi_strong["Target"])
node_ids_100 = set(nodes_gephi_strong["Id"])

print("Threshold 30 unmatched nodes:", len(edge_nodes_30 - node_ids_30))
print("Threshold 100 unmatched nodes:", len(edge_nodes_100 - node_ids_100))

Threshold 30 unmatched nodes: 0
Threshold 100 unmatched nodes: 0


The validation confirms that every product used in the edge files exists in the matching node files. Both unmatched counts are 0, which means the threshold-30 and threshold-100 node-edge pairs are correctly aligned and ready for Gephi import.